In [ ]:
# Installing dependencies
!pip install transformers datasets gradio torch accelerate -q

In [ ]:
# Importing Libraries
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [ ]:
# Expanded empathetic dataset (50 pairs for better training)
data = [
    # Anxiety & Sleep
    {"conv_id": 1,  "utterance": "I've been feeling really anxious lately and can't sleep"},
    {"conv_id": 1,  "utterance": "I'm so sorry to hear that. Anxiety can be really exhausting. What's been keeping you up at night?"},
    {"conv_id": 2,  "utterance": "I keep waking up at 3am with a racing heart"},
    {"conv_id": 2,  "utterance": "That sounds really frightening. Nighttime anxiety can feel so isolating. Have you tried any breathing exercises before bed?"},
    {"conv_id": 3,  "utterance": "I have a panic attack almost every morning"},
    {"conv_id": 3,  "utterance": "Panic attacks are so overwhelming and I'm truly sorry you're going through this daily. You deserve support. Have you been able to speak with a doctor?"},

    # Loneliness & Isolation
    {"conv_id": 4,  "utterance": "I feel like nobody understands what I'm going through"},
    {"conv_id": 4,  "utterance": "That feeling of isolation is really painful. I want you to know that I'm here and I truly want to understand you."},
    {"conv_id": 5,  "utterance": "I have no one to talk to about my problems"},
    {"conv_id": 5,  "utterance": "I'm glad you're here and talking now. You don't have to carry this alone. What's been weighing on your heart?"},
    {"conv_id": 6,  "utterance": "I moved to a new city and I feel completely alone"},
    {"conv_id": 6,  "utterance": "Starting over in a new place is genuinely hard. The loneliness you feel is real. Give yourself grace — it takes time to build connections."},

    # Overwhelm & Stress
    {"conv_id": 7,  "utterance": "Everything feels overwhelming right now"},
    {"conv_id": 7,  "utterance": "It's okay to feel overwhelmed. Let's take it one step at a time. Can you tell me what's weighing on you most?"},
    {"conv_id": 8,  "utterance": "I have too much to do and I don't know where to start"},
    {"conv_id": 8,  "utterance": "When everything piles up it can feel paralyzing. Let's slow down together. What is the one smallest thing you could do right now?"},
    {"conv_id": 9,  "utterance": "I'm stressed about my exams and feel like I'll fail"},
    {"conv_id": 9,  "utterance": "Exam stress is so real and valid. It's clear you care about doing well. Let's talk about what's worrying you most — sometimes naming it helps."},

    # Sadness & Low Mood
    {"conv_id": 10, "utterance": "I feel so sad and I don't know why"},
    {"conv_id": 10, "utterance": "Sometimes sadness comes without a clear reason, and that's okay. I'm here with you. You don't have to explain it."},
    {"conv_id": 11, "utterance": "I've been crying a lot today for no reason"},
    {"conv_id": 11, "utterance": "Sometimes our emotions need an outlet even without a specific cause. Crying can be healing. I'm right here with you."},
    {"conv_id": 12, "utterance": "I don't feel happy anymore doing the things I used to love"},
    {"conv_id": 12, "utterance": "Losing interest in things you once loved is a painful experience and can be a sign you need extra support. Have you been able to rest and take care of yourself?"},

    # Self-Worth & Confidence
    {"conv_id": 13, "utterance": "I feel like I'm not good enough"},
    {"conv_id": 13, "utterance": "I hear you, and I want you to know that feeling doesn't reflect the truth. You are enough, just as you are."},
    {"conv_id": 14, "utterance": "I always compare myself to others and feel like a failure"},
    {"conv_id": 14, "utterance": "Comparison can be so cruel to ourselves. Your journey is uniquely yours and your worth isn't measured by anyone else's progress."},
    {"conv_id": 15, "utterance": "I feel like I'm a burden to everyone around me"},
    {"conv_id": 15, "utterance": "You are not a burden. The people who care about you want you in their lives. That thought, though painful, isn't the truth."},

    # Relationships
    {"conv_id": 16, "utterance": "I had a fight with my best friend and I feel terrible"},
    {"conv_id": 16, "utterance": "Conflicts with close friends can hurt deeply. It shows how much the friendship means to you. How are you feeling about it now?"},
    {"conv_id": 17, "utterance": "My parents don't listen to me and I feel ignored"},
    {"conv_id": 17, "utterance": "Feeling unheard by the people closest to us is really painful. Your feelings and thoughts absolutely matter."},
    {"conv_id": 18, "utterance": "I just went through a breakup and I feel heartbroken"},
    {"conv_id": 18, "utterance": "Heartbreak is one of the most real kinds of pain. Please be gentle with yourself right now. What do you need most today?"},

    # Work & Life Pressure
    {"conv_id": 19, "utterance": "I had a terrible day at work, my boss yelled at me"},
    {"conv_id": 19, "utterance": "That sounds really stressful and hurtful. Being yelled at is never okay. How are you feeling now?"},
    {"conv_id": 20, "utterance": "I'm afraid of losing my job"},
    {"conv_id": 20, "utterance": "Job insecurity is incredibly stressful and that fear is completely understandable. You're not alone in feeling this way. Let's talk through what's happening."},
    {"conv_id": 21, "utterance": "I feel burnt out and exhausted from work"},
    {"conv_id": 21, "utterance": "Burnout is real and it's your mind and body asking for a break. It's okay to rest. What does rest look like for you?"},

    # General Emotional Support
    {"conv_id": 22, "utterance": "I just need someone to talk to"},
    {"conv_id": 22, "utterance": "I'm right here and I'm listening. Take your time. What's on your mind?"},
    {"conv_id": 23, "utterance": "I feel like things will never get better"},
    {"conv_id": 23, "utterance": "When we're in pain it can feel permanent, but feelings do shift. I believe things can get better for you. What's one small thing that brought you comfort recently?"},
    {"conv_id": 24, "utterance": "I'm scared about the future"},
    {"conv_id": 24, "utterance": "Uncertainty about the future is something many people feel deeply. That fear makes sense. You don't have to figure it all out right now."},
    {"conv_id": 25, "utterance": "I've been having dark thoughts lately"},
    {"conv_id": 25, "utterance": "Thank you for trusting me with that. Dark thoughts can feel very isolating. I want to make sure you're safe — are you having thoughts of hurting yourself?"},
]

train_df = pd.DataFrame(data)
valid_df = pd.DataFrame(data[:12])

print("Dataset ready!")
print(f"Train rows: {len(train_df)}, Valid rows: {len(valid_df)}")
print(train_df.head(4))

Dataset ready!
Train rows: 50, Valid rows: 12
   conv_id                                          utterance
0        1  I've been feeling really anxious lately and ca...
1        1  I'm so sorry to hear that. Anxiety can be real...
2        2        I keep waking up at 3am with a racing heart
3        2  That sounds really frightening. Nighttime anxi...


In [ ]:
# Loading Model & Tokenizer
MODEL_NAME = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))

print(f"Model loaded: {MODEL_NAME}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: microsoft/DialoGPT-small
   Parameters: 124,439,808


In [ ]:
# Building training pairs
def build_pairs(df):
    pairs = []
    grouped = df.groupby('conv_id')
    for conv_id, group in grouped:
        group = group.reset_index(drop=True)
        utterances = group['utterance'].tolist()
        for i in range(0, len(utterances) - 1, 2):
            user_msg = str(utterances[i]).strip()
            bot_msg  = str(utterances[i+1]).strip() if i+1 < len(utterances) else ""
            if user_msg and bot_msg:
                pairs.append({
                    "text": f"User: {user_msg}\nChatbot: {bot_msg}{tokenizer.eos_token}"
                })
    return pairs

train_pairs = build_pairs(train_df)
valid_pairs = build_pairs(valid_df)

print(f"Train pairs: {len(train_pairs)}")
print(f"Valid pairs: {len(valid_pairs)}")
print("\nSample:")
print(train_pairs[0]['text'])

Train pairs: 25
Valid pairs: 6

Sample:
User: I've been feeling really anxious lately and can't sleep
Chatbot: I'm so sorry to hear that. Anxiety can be really exhausting. What's been keeping you up at night?<|endoftext|>


In [ ]:
# Tokenizing dataset
train_data = Dataset.from_list(train_pairs)
valid_data = Dataset.from_list(valid_pairs)

def tokenize(example):
    tokens = tokenizer(
        example['text'],
        truncation=True,
        max_length=128,
        padding='max_length',
    )
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

tokenized_train = train_data.map(tokenize, batched=False, remove_columns=['text'])
tokenized_val   = valid_data.map(tokenize, batched=False, remove_columns=['text'])

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f"Tokenized — Train: {len(tokenized_train)}, Val: {len(tokenized_val)}")

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenized — Train: 25, Val: 6


In [ ]:
# Training the model
training_args = TrainingArguments(
    output_dir="./mental_health_bot",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=False,
    fp16_full_eval=False,
    report_to="none",
    learning_rate=5e-5,
    optim='adafactor',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("Strating the Training")
trainer.train()
print("Training complete!")

trainer.save_model("./mental_health_bot_final")
tokenizer.save_pretrained("./mental_health_bot_final")
print("Model saved to ./mental_health_bot_final")

Strating the Training


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,7.489065,5.516869
2,6.687556,4.444469
3,5.319067,3.961622
4,5.026800,3.621414
5,4.452776,3.580359


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Training complete!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./mental_health_bot_final


In [ ]:
#  A Response generation function

CRISIS_KEYWORDS = [
    'suicide', 'kill myself', 'end my life', 'harm myself',
    'want to die', 'better off dead', 'self harm', 'hurt myself'
]

CRISIS_RESPONSE = (
    "💙 I hear you and I'm really concerned about your safety right now. "
    "Please reach out to a crisis helpline immediately:\n"
    "• **Pakistan**: Umang helpline — 0317-4288665\n"
    "• **International**: iCall — icallhelpline.org\n"
    "You are not alone and help is available 24/7."
)

def generate_response(user_input: str) -> str:
    """Generate a chatbot response for the given user input."""

    # 1. Crisis check
    if any(kw in user_input.lower() for kw in CRISIS_KEYWORDS):
        return CRISIS_RESPONSE

    # 2. Building prompt
    prompt = f"User: {user_input}\nChatbot:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 3. Generating
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.75,
            top_p=0.92,
            top_k=50,
            repetition_penalty=1.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 4. Decoding only the new tokens
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response  = tokenizer.decode(generated, skip_special_tokens=True)
    response  = response.split('\n')[0].strip()

    return response if response else "I'm here for you. Can you share more about how you're feeling?"


print("Test response:")
print(generate_response("I've been feeling really anxious lately"))

Test response:
How's it feel to be in control of your thoughts and emotions?


In [ ]:
# Gradio Chat Interface

GREETING = "Hello! 💙 I'm your Mental Health Support companion. I'm here to listen without judgment. How are you feeling today?"

def chat(user_message: str, history: list):
    """
    Gradio chat function.
    history: list of [user_msg, bot_msg] pairs (Gradio format)
    """
    if not user_message.strip():
        return "", history

    bot_reply = generate_response(user_message.strip())
    history.append([user_message, bot_reply])
    return "", history


# ── UI ──────────────────────────────────────────────────────────────
with gr.Blocks(
    title="🧠 Mental Health Support Chatbot",
    theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate"),
    css="""
        body { font-family: 'Segoe UI', sans-serif; }
        .title-box { text-align: center; padding: 10px 0; }
        .disclaimer { font-size: 12px; color: #888; text-align: center; }
        footer { display: none !important; }
    """
) as demo:

    # Header
    gr.HTML("""
    <div class="title-box">
        <h1>🧠 Mental Health Support Chatbot</h1>
        <p style="color:#555">A safe space to share how you're feeling. I'm here to listen. 💙</p>
    </div>
    """)

    # Chat window
    chatbot = gr.Chatbot(
        value=[[None, GREETING]],
        label="Conversation",
        height=420,
        bubble_full_width=False,
        avatar_images=(None, "https://i.imgur.com/7yUvePI.png"),  # bot avatar
    )

    # Input row
    with gr.Row():
        msg_box = gr.Textbox(
            placeholder="Type how you're feeling...",
            show_label=False,
            scale=5,
            container=False,
        )
        send_btn = gr.Button("Send 💬", variant="primary", scale=1)

    # Suggested starters
    gr.HTML("<p style='margin:8px 0 4px; color:#666; font-size:13px;'>💡 Try asking:</p>")
    with gr.Row():
        ex1 = gr.Button("I feel really anxious",         size="sm")
        ex2 = gr.Button("Everything feels overwhelming", size="sm")
        ex3 = gr.Button("I feel like I'm not good enough", size="sm")
        ex4 = gr.Button("I just need someone to talk to", size="sm")

    # Clear button
    clear_btn = gr.Button("🔄 Start New Conversation", variant="secondary")

    # Crisis disclaimer
    gr.HTML("""
    <p class="disclaimer">
    ⚠️ This chatbot is NOT a substitute for professional mental health care.
    If you are in crisis, please contact a helpline immediately.
    Pakistan: Umang — 0317-4288665
    </p>
    """)

    # ── Event bindings ────────────────────────────────────────────────
    msg_box.submit(chat, [msg_box, chatbot], [msg_box, chatbot])
    send_btn.click(chat,  [msg_box, chatbot], [msg_box, chatbot])

    # Example buttons fill textbox
    for btn, text in [
        (ex1, "I feel really anxious"),
        (ex2, "Everything feels overwhelming"),
        (ex3, "I feel like I'm not good enough"),
        (ex4, "I just need someone to talk to"),
    ]:
        btn.click(lambda t=text: t, outputs=msg_box)

    clear_btn.click(
        lambda: [[None, GREETING]], outputs=chatbot
    )

# Launch
demo.launch(share=True)   # share=True gives a public URL (useful for Colab/Kaggle)

/tmp/ipykernel_17340/2589137612.py:19: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_17340/2589137612.py:19: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_17340/2589137612.py:39: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_17340/2589137612.py:39: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1734

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ddb1ff9122d65e46c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
